In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path_riddles = "/content/drive/MyDrive/PercorsoEccellenza/riddles/"

In [ ]:
!git clone https://github.com/GiovanniAdelfio/small_LM
%cd small_LM

Cloning into 'small_LM'...
remote: Enumerating objects: 476, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 476 (delta 59), reused 11 (delta 11), pack-reused 362 (from 2)
Receiving objects: 100% (476/476), 10.69 MiB | 15.37 MiB/s, done.
Resolving deltas: 100% (219/219), done.
/content/small_LM


In [ ]:
%cd updated

[Errno 2] No such file or directory: 'updated'
/content/small_LM


In [ ]:
## required imports

import pickle
import torch
import torch.optim as optim
import copy
import random
import os

random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
path = os.getcwd() + os.sep + "files" + os.sep
path_checkpoints = os.getcwd() + os.sep + "checkpoints" + os.sep

def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  Nello __init__ di GPTModel, alla fine, aggiungi:
  self.apply(self._init_weights)

## Dataset riddles

In [1]:
!pip install datasets pandas

   ---------------------------------------- 0.0/526.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/526.6 kB ? eta -:--:--
   ---------------------------------------- 526.6/526.6 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.5 MB 2.8 MB/s eta 0:00:10
   - -------------------------------------- 1.3/27.5 MB 2.5 MB/s eta 0:00:11
   -- ------------------------------------- 1.8/27.5 MB 2.3 MB/s eta 0:00:12
   --- ------------------------------------ 2.4/27.5 MB 2.3 MB/s eta 0:00:11
   ---- ----------------------------------- 2.9/27.5 MB 2.3 MB/s eta 0:00:11
   ---- ----------------------------------- 3.1/27.5 MB 2.4 MB/s eta 0:00:11
   ---- ----------------------------------- 3.4/27.5 MB 2.1 MB/s eta 0:00:12
   ----- ---------------------------------- 3.9/27.5 MB 2.2 MB/s eta 0:00:11
   ------ -------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset
import pandas as pd
import random

# 1. Carichiamo il dataset specifico
print("Sto scaricando Hypersniper/riddles_v1...")
dataset = load_dataset("Hypersniper/riddles_v1", split="train")
df = dataset.to_pandas()

c:\Users\Giovanni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sto scaricando Hypersniper/riddles_v1...


c:\Users\Giovanni\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Giovanni\.cache\huggingface\hub\datasets--Hypersniper--riddles_v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 469/469 [00:00<00:00, 10253.53 examples/s]

In [3]:
df

,answer,output,instruction
0,music,The question seems to be a riddle that is focu...,"At the sound of me, one may dream or stamp the..."
1,onion,The answer to this riddle is an onion. Let's b...,"Take off my skin, I won't cry, but you will."
2,fleas,This riddle is an interesting one because it u...,"When they are caught, they are thrown away. Wh..."
3,eye,"The answer to this question is ""eye."" Let's br...","What is pronounced like one letter, written wi..."
4,sun,"The question is a riddle, and the solution to ...",What goes through the door without pinching it...
...,...,...,...
464,The letter “g”.,"To answer your question, we need to break it d...",What is at the end of everything?
465,Lice.,The question is a riddle that deals with the c...,"What we caught, we threw away. We kept what we..."
466,The word “ton”.,"Sure, I'd be happy to help you figure out this...",Forward I am heavy but backward I am not. What...
467,A garbage truck.,"The question at hand seems to be a riddle, ask...",What has four wheels and flies?


In [4]:
# 2. Funzione per creare il formato "Gioco"
def prepara_sfida(row):
    # La risposta (l'oggetto)
    risposta = row['answer']

    # L'indovinello
    enigma = row['instruction']

    # Creiamo un prompt vario per l'utente
    prompts = [f"A riddle about {risposta}:",
               f"Can you solve this riddle about {risposta}?",
               f"Here is a riddle about {risposta}:",
               f"Riddle time! Topic: {risposta}.",
               f"Try to guess this riddle about {risposta}"
    ]
    user_input = random.choice(prompts)

    # Costruiamo la risposta dell'AI (Indovinello + Spoiler nascosto)
    ai_output = f"RIDDLE: {enigma} \n ANSWER: {risposta}"

    return f"[BOS] {user_input} \n {ai_output} [EOS]"

In [5]:
def prepara_con_spiegazione(row):
    risposta = row['answer']
    indovinello = row['instruction']
    spiegazione = row['output']

    prompts = [
        f"Write a riddle about {risposta}",
        f"Give me a puzzle regarding {risposta}",
        f"Describe {risposta} with a riddle",
        f"Here is a riddle about {risposta}:",
    ]
    user_input = random.choice(prompts)

    # Costruiamo l'output strutturato
    ai_output = (
        f"RIDDLE: {indovinello} \n ANSWER: {risposta} \n WHY? {spiegazione}"
    )

    return f"[BOS] {user_input} \n {ai_output} [EOS]"

In [11]:
# 3. Applichiamo la funzione
# Filtriamo eventuali righe vuote per sicurezza
df = df.dropna(subset=['answer', 'output'])
df['text'] = df.apply(prepara_sfida, axis=1)

# 4. Vediamo il risultato
print(f"\nHo preparato {len(df)} indovinelli pronti per il training.")
print("--- ESEMPIO ---")
print(df['text'].iloc[0])


Ho preparato 469 indovinelli pronti per il training.
--- ESEMPIO ---
[BOS] Try to guess this riddle about music 
 RIDDLE: At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep. 
 ANSWER: music [EOS]


In [12]:
df["text"].iloc[0]

'[BOS] Try to guess this riddle about music \n RIDDLE: At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep. \n ANSWER: music [EOS]'

In [13]:
lines = []
for i in range(len(df)):
  lines.append(df["text"].iloc[i])


txt_chr = "".join(lines[:-1])

## Dataset Tinystories


In [ ]:
from datasets import load_dataset

# 1. Carica TinyStories (solo il train split)
# streaming=True permette di non scaricare GB di dati tutti insieme
print("Carico TinyStories...")
dataset_general = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# 2. Preleviamo un campione gestibile (es. 50.000 storie) per il pre-training
# Questo sarà il tuo "libro di grammatica"
data_iter = iter(dataset_general)
stories = []

print("Sto scaricando 50k storie...")
for _ in range(50000):
    row = next(data_iter)
    stories.append(row['text'])

# 3. Salva in un file per addestrare il Tokenizer e il Modello
with open("tinystories_train.txt", "w", encoding="utf-8") as f:
    for story in stories:
        # Aggiungi i token speciali anche qui!
        f.write(f"[BOS] {story} [EOS]\n")

print("Fatto! File 'tinystories_train.txt' pronto per il training.")

Carico TinyStories...


README.md: 0.00B [00:00, ?B/s]

Sto scaricando 50k storie...
Fatto! File 'tinystories_train.txt' pronto per il training.


## Data fetching

Selecting only "Ordinary life" dialogues.

In [ ]:
used_lines = []
allow_all = False

group = 1
with open(path + "dialogues_topic.txt", encoding="utf-8") as topic:
  for i, line in enumerate(topic):
    if int(line) == group:
      used_lines += [i]

lines = []

with open(path + "dialogues_text.txt", encoding="utf-8") as txt:
  for i, el in enumerate(txt):
    if not allow_all and (i not in used_lines):
      continue
    lines.append(el)

Choosing "@" as a token for the end of a person's sentence in the dialogue, and cleaning the sentences.

We then concatenate the entire dataset into a single string: txt_chr.

In [ ]:
for i, el in enumerate(lines):
  lines[i] = el.replace("\n", " ")
  lines[i] = lines[i].replace("__eou__", "@")

txt_chr = "".join(lines[:-1])

In [ ]:
j=0
for i in range(len(lines)):
  j+= lines[i].count("@")
print(f"Averege number of turns per dialog: {j//len(lines)}")

Averege number of turns per dialog: 8


In [ ]:
j=0
for i in range(len(lines)):
  j+= lines[i].count("00")
  if j>1000:
    print(lines[i])
    break
print(f"Averege number of turns per dialog: {j}, {len(lines)}")

Averege number of turns per dialog: 304, 3646


In [ ]:
(lines[0] + "0").count()

TypeError: count() takes at least 1 argument (0 given)

## Implementing Tokenizers' tokenizer


In [ ]:
dataset_file = "riddles_spiegazione.txt"

with open(dataset_file, "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

print(f"Dati salvati in {dataset_file}")

Dati salvati in riddles_spiegazione.txt


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors

# 1. Inizializza il Tokenizer vuoto (Modello BPE)
tokenizer = Tokenizer(models.BPE())

# 2. Pre-tokenizzazione: Come spezzare le parole prima del BPE?
# ByteLevel è ottimo: gestisce spazi e caratteri strani convertendoli in byte.
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# 4. Configura il Trainer
trainer = trainers.BpeTrainer(
    vocab_size=4000,
    min_frequency=2, # Ignora token che appaiono meno di 2 volte
    special_tokens=["[PAD]", "[BOS]", "[EOS]", "[SEP]"], # Token speciali utili
    show_progress=True
)

# 5. Avvia l'addestramento sul file creato prima
print("Inizio training del tokenizer...")
tokenizer.train(["riddles.txt"], trainer)

# 6. Salva il tokenizer in un file JSON
tokenizer.save(path + "custom_tokenizer.json")
print("Tokenizer salvato come 'custom_tokenizer.json'")

Inizio training del tokenizer...
Tokenizer salvato come 'custom_tokenizer.json'


In [ ]:
path_riddles

'/content/drive/MyDrive/PercorsoEccellenza/riddles/'

In [16]:
from tokenizers import Tokenizer

# Carica dal file salvato
tokenizer = Tokenizer.from_file(path_riddles + "custom_tokenizer_tinystories.json")

In [ ]:
tokenizer.add_tokens(["ANSWER:"])
tokenizer.add_tokens(["RIDDLE:"])

1

In [ ]:
text = lines[0]

# encode() restituisce un oggetto Encoding con molte info
encoded = tokenizer.encode(text)

# Quello che ti serve sono gli ids
ids = encoded.ids

print(f"Input: {text}")
print(f"IDs: {ids}")

decoded_text = tokenizer.decode(ids)

print(f"Output: {decoded_text}")

Input: [BOS] Here is a riddle about music: 
 RIDDLE: At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep. 
 ANSWER: music [EOS]
IDs: [1, 3498, 269, 115, 187, 2542, 527, 1475, 27, 265, 104, 4001, 2690, 119, 974, 203, 387, 13, 495, 3621, 1250, 589, 3257, 312, 2224, 13, 2690, 119, 974, 203, 387, 13, 495, 3621, 1258, 589, 1412, 222, 455, 15, 265, 104, 4000, 1475, 104, 2]
Output:  Here is a riddle about music: 
 RIDDLE: At the sound of me, one may dream or stamp their feet, At the sound of me, one may laugh or sometimes weep. 
 ANSWER: music 


In [17]:
dataset = []

for line in lines:
    encoded = tokenizer.encode(line)
    ids = encoded.ids
    dataset.append(ids)

In [ ]:
vocab_size = tokenizer.get_vocab_size()
print(f"Dimensione Totale Vocabolario: {vocab_size}")

print("\n--- ULTIMI 30 TOKENS ---")
for id in range(vocab_size - 20, vocab_size):
    token = tokenizer.id_to_token(id)
    print(f"ID {id}: {tokenizer.decode([id])}")

Dimensione Totale Vocabolario: 4000

--- ULTIMI 30 TOKENS ---
ID 3980:  surrender
ID 3981: cast
ID 3982:  pipe
ID 3983:  yach
ID 3984:  spir
ID 3985:  breat
ID 3986:  lions
ID 3987:  blowing
ID 3988: come
ID 3989: io
ID 3990:  sle
ID 3991:  ped
ID 3992: urb
ID 3993:  marched
ID 3994:  disturb
ID 3995:  speak
ID 3996:  turkey
ID 3997:  toilet
ID 3998: Bye
ID 3999: aked


In [ ]:
# Trova l'ID del padding (ti servirà nel collate_fn del DataLoader)
pad_token_id = tokenizer.token_to_id("[PAD]")
vocab_size = tokenizer.get_vocab_size()
_turn_token_id = tokenizer.token_to_id("@")

print(f"Vocab Size: {vocab_size}")
print(f"PAD ID: {pad_token_id}")
print(f"Turn token ID: {_turn_token_id}")

Vocab Size: 4000
PAD ID: 0
Turn token ID: None


In [ ]:
compression_ratio = 0
for comp_line, line in zip(dataset, lines):
    compression_ratio += len(comp_line) / len(line)
print(f"Compression Ratio: {100*(1-(compression_ratio / len(lines))):.4f}%")

Compression Ratio: 99.9416%


In [ ]:
print(dataset[0])
print(tokenizer.decode(dataset[0]))

[28, 81, 449, 314, 60, 2901, 200, 3929, 2393, 15, 3716, 9, 182, 3232, 12, 182, 201, 341, 12, 535, 182, 1257, 678, 12, 256, 1015, 547, 200, 1533, 232, 647, 598, 247, 200, 603, 232, 256, 432, 359, 247, 12, 647, 585, 325, 200, 577, 232, 256, 555, 521, 200, 506, 232, 256, 1920, 200, 1849, 1118, 521, 232, 429, 182, 306, 641, 12, 1688, 1717, 311, 452, 314, 60, 129, 451, 453, 399, 442, 226, 164, 302, 437, 238, 172, 444, 382, 26, 2901, 200, 3929, 2393, 15, 3716, 9, 182, 3232, 12, 182, 201, 341, 12, 535, 182, 1257, 678, 12, 256, 1015, 547, 200, 1533, 232, 647, 598, 247, 200, 603, 232, 256, 432, 359, 247, 12, 647, 585, 325, 200, 577, 232, 256, 555, 521, 200, 506, 232, 256, 1920, 200, 1849, 1118, 521, 232, 429, 182, 306, 641, 14, 402, 302, 450, 164, 1688, 1717, 311, 14, 129, 411, 482, 491, 172, 875, 2901, 363, 12, 3594, 13, 81, 270, 172, 376, 2490, 181, 2041, 14, 1832, 845, 1214, 172, 1540, 306, 172, 1540, 1185, 553, 172, 2550, 80, 378, 81, 14, 486, 172, 376, 2490, 201, 172, 624, 774, 238, 516, 2

## Creating datasets and dataloaders

We now create the target dataset from our inputs, by associating for each sequence of context_size lenght, the corresponding sequence in the text translated by one token.

We now divide the dataset in train, verification and test. We also trasform our datasets and targets into torch tensors.

In [ ]:
from utils.data import split
train_dataset, val_dataset, test_dataset = split(dataset, t=0.7, v=0.2, seed=42, to_torch = True, device = device)

For consistency we save the randomly generated splits.

In [ ]:
os.chdir(path)
torch.save([train_dataset, val_dataset,test_dataset], "dataset.pt")
os.chdir("..")

In [ ]:
os.chdir(path)
train_dataset, val_dataset, test_dataset = torch.load("dataset.pt", weights_only= "True")
os.chdir("..")

We create a dataset and dataloader using pytorch utils, and wrap it on our files.

In [ ]:
from torch.utils.data import DataLoader
from utils.data import SLM_dataset
bs = 32
cs = 128

train = SLM_dataset(train_dataset, cs)
val = SLM_dataset(val_dataset, cs)

train_dataloader = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val, batch_size=bs, shuffle=False, num_workers=0)

/content/small_LM/utils/data.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks.append(torch.tensor(padding_mask, dtype=torch.bool))


## Downloads

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(path_riddles + "custom_tokenizer_tinystories.json")

In [ ]:
train_dataset, val_dataset, test_dataset = torch.load(path_riddles + "dataset_tinystories.pt", weights_only= "True")

In [ ]:
from torch.utils.data import DataLoader
from utils.data import SLM_dataset
bs = 32
cs = 128

train = SLM_dataset(train_dataset, cs)
val = SLM_dataset(val_dataset, cs)

train_dataloader = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val, batch_size=bs, shuffle=False, num_workers=0)

/content/small_LM/utils/data.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  masks.append(torch.tensor(padding_mask, dtype=torch.bool))


## Finetuning

In [ ]:
used_lines_ft = []

group = 10
with open(path + "dialogues_topic.txt", encoding="utf-8") as topic:
  for i, line in enumerate(topic):
    if int(line) == group:
      used_lines_ft += [i]
lines_ft = []

with open(path + "dialogues_text.txt", encoding="utf-8") as txt:
  for i, el in enumerate(txt):
    if i not in used_lines_ft:
      continue
    lines_ft.append(el)

In [ ]:
for i, el in enumerate(lines_ft):
  lines_ft[i] = el.replace("\n", " ")
  lines_ft[i] = lines_ft[i].replace("__eou__", "@")

txt_chr_ft = "".join(lines_ft[:-1])

In [ ]:
dataset_file = "daily_dialog_finetuning.txt"

with open(dataset_file, "w", encoding="utf-8") as f:
    for line in lines_ft:
        f.write(line + "\n")

print(f"Dati salvati in {dataset_file}")

Dati salvati in daily_dialog_finetuning.txt


In [ ]:
dataset_ft = []

for line in lines_ft:
    encoded = tokenizer.encode(line)
    ids = encoded.ids
    dataset_ft.append(ids)

In [ ]:
from utils.data import split
train_dataset, val_dataset, test_dataset = split(dataset_ft, t=0.7, v=0.2, seed=42, to_torch = True, device = device)

In [ ]:
from torch.utils.data import DataLoader
from utils.data import SLM_dataset
bs = 32
cs = 128

train = SLM_dataset(train_dataset, cs)
val = SLM_dataset(val_dataset, cs)

train_dataloader_ft = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0)
val_dataloader_ft = DataLoader(val, batch_size=bs, shuffle=False, num_workers=0)

## Model Training via Lightning


We import our model, and generation function. We then initialize the model.

In [ ]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 71.5 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.ao.quantization
import lightning.pytorch as pl
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout=0.1): # Aggiungi parametro dropout
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout) # <--- CRUCIALE: Dropout qui
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, dr = 0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=n_embd,
            num_heads=n_head,
            batch_first=True,
            dropout=dr
        )
        self.ffwd = FeedFoward(n_embd, dropout=dr)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.dropout1 = nn.Dropout(dr)
        self.dropout2 = nn.Dropout(dr)
        self._causal_mask_cache = {}  # ✅ cache per size

    def _get_causal_mask(self, size, device):
        """Return cached causal mask — only built once per sequence length."""
        if size not in self._causal_mask_cache:
            self._causal_mask_cache[size] = torch.triu(
                torch.ones(size, size), diagonal=1
            ).bool().to(device)
        return self._causal_mask_cache[size]

    def forward(self, x, padding_mask=None):
        x_norm = self.ln1(x)

        attn_mask = self._get_causal_mask(x_norm.size(1), x_norm.device)  
            
        if padding_mask is None:
            x_attn, _ = self.mha(
                x_norm, x_norm, x_norm,
                attn_mask=attn_mask,
                need_weights=False
            )
        else:
            x_attn, _ = self.mha(
                x_norm, x_norm, x_norm,
                attn_mask=attn_mask,
                need_weights=False,
                key_padding_mask=padding_mask
            )

        x = x + self.dropout1(x_attn)
        x_norm = self.ln2(x)
        x_ffwd = self.ffwd(x_norm)
        x = x + self.dropout2(x_ffwd)

        return x


class GPTModel(nn.Module):
    def __init__(self, block_size, vocab_size, n_embd, n_head, n_layer, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(self.block_size, n_embd)
        self.emb_dropout = nn.Dropout(dropout) # <--- CRUCIALE: Dropout sugli embedding

        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dr=dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None, padding_mask=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_indices = torch.arange(T, device=idx.device)  # still needed, but...
        pos_emb = self.position_embedding_table(pos_indices)
        x = tok_emb + pos_emb
        x = self.emb_dropout(x)

        for block in self.blocks:
            x = block(x, padding_mask=padding_mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_view = logits.view(B * T, C)
            targets_view = targets.view(B * T)
            loss = F.cross_entropy(logits_view, targets_view, ignore_index=-100)

        return logits, loss

class GPTLightningModule(pl.LightningModule):
    def __init__(
        self,
        block_size,
        vocab_size,
        n_embd,
        n_head,
        n_layer,
        learning_rate=3e-4,
        weight_decay=0.1,
        max_epochs=100,
        use_qat=False,
        qat_backend='fbgemm',
        dropout=0.1
    ):
        """
        PyTorch Lightning wrapper for GPT model with optional QAT.

        Args:
            block_size: Maximum sequence length
            vocab_size: Size of vocabulary
            n_embd: Embedding dimension
            n_head: Number of attention heads
            n_layer: Number of transformer blocks
            learning_rate: Learning rate for optimizer
            weight_decay: Weight decay for optimizer
            max_epochs: Maximum number of training epochs (for scheduler)
            use_qat: Whether to use Quantization-Aware Training
            qat_backend: Backend for quantization ('fbgemm' for x86, 'qnnpack' for ARM)
        """
        super().__init__()
        self.save_hyperparameters()

        # Create the model
        self.model = GPTModel(block_size, vocab_size, n_embd, n_head, n_layer, dropout)

        # QAT setup
        self.use_qat = use_qat
        if self.use_qat:
            self.model.qconfig = torch.ao.quantization.get_default_qat_qconfig(qat_backend)
            # Prepare model for QAT
            torch.ao.quantization.prepare_qat(self.model, inplace=True)

    def forward(self, idx, targets=None, padding_mask=None):
        return self.model(idx, targets, padding_mask)

    def training_step(self, batch, batch_idx):
        # Assuming batch is a tuple of (input_ids, targets) or dict
        idx = batch['x']
        targets = batch['y']
        padding_mask = batch.get('padding_mask', None)

        logits, loss = self(idx, targets, padding_mask)

        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        idx = batch['x']
        targets = batch['y']
        padding_mask = batch.get('padding_mask', None)

        logits, loss = self(idx, targets, padding_mask)

        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        idx = batch['x']
        targets = batch['y']
        padding_mask = batch.get('padding_mask', None)

        logits, loss = self(idx, targets, padding_mask)

        self.log('test_loss', loss, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):

        # Separa i parametri: quelli con decay e quelli senza
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # Filtra quelli che non richiedono gradiente
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}

        # Tutti i parametri 2D (pesi matrici) avranno weight decay
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        # Tutti i bias e layernorm (1D) non avranno weight decay
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
        {'params': decay_params, 'weight_decay': self.hparams.weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]

        # Create optimizer with weight decay
        optimizer = AdamW(
            optim_groups,
            lr=self.hparams.learning_rate,
            betas=(0.9, 0.99) # Standard GPT betas
        )

        # Create learning rate scheduler
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.learning_rate * 0.1
        )

        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1
            }
        }

    def convert_to_quantized(self):
        """
        Convert the QAT model to a fully quantized model.
        Call this after training is complete.
        """
        if not self.use_qat:
            raise ValueError("Model was not trained with QAT. Set use_qat=True during initialization.")

        self.model.eval()
        torch.ao.quantization.convert(self.model, inplace=True)
        return self.model

    # In GPTLightningModule.generate(), replace the loop with:
def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, 
             padding_mask=None, repetition_penalty=1.0):
    self.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = (idx if idx.size(1) <= self.model.block_size 
                       else idx[:, -self.model.block_size:])

            logits, _ = self(idx_cond, padding_mask=padding_mask)
            logits = logits[:, -1, :]  # (B, vocab_size)

            if repetition_penalty != 1.0:
                token_visti = set(idx[0].tolist())
                for token_id in token_visti:
                    if logits[0, token_id] < 0:
                        logits[0, token_id] *= repetition_penalty
                    else:
                        logits[0, token_id] /= repetition_penalty

            logits = logits / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

    return idx


In [ ]:
#from model.lightning_model import GPTLightningModule
import lightning.pytorch as pl


model = GPTLightningModule(
    block_size=128,
    vocab_size=4000,
    n_embd=256,
    n_head=8,
    n_layer=6,
    use_qat=False,  # Default
    learning_rate = 1e-3,
    dropout = 0.1,
    weight_decay=1e-1
)

In [ ]:
from model.lightning_model import GPTLightningModule
print("Caricamento del modello scelto per ulteriore training...")
model = GPTLightningModule.load_from_checkpoint(path_riddles + "/gpt-epoch=09-val_loss=2.0858.ckpt",
    learning_rate = 5e-5,
    dropout = 0.1,
    weight_decay=1e-2
    )
model.to("cuda" if torch.cuda.is_available() else "cpu");

Caricamento del modello scelto per ulteriore training...


In [ ]:
import torch
#from model.lightning_model import GPTLightningModule

print("Caricamento del modello scelto per ulteriore training...")


# 2. Carica il file .ckpt usando PyTorch base (mappando su CPU per evitare problemi di memoria)
percorso_file = path_riddles + "/40_epoche.ckpt"
checkpoint = torch.load(percorso_file, map_location=torch.device('cpu'))

# 3. Inserisci i pesi nel modello
# Controlliamo se i pesi sono salvati sotto la chiave 'state_dict' o se il file è già il dizionario dei pesi
try:
    if 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'], strict=False)
    else:
        model.load_state_dict(checkpoint, strict=False)
    print("Pesi caricati con successo!")
except Exception as e:
    print(f"Errore durante il caricamento dei pesi: {e}")

# 4. Sposta il modello su GPU se disponibile
model.to("cuda" if torch.cuda.is_available() else "cpu")

Caricamento del modello scelto per ulteriore training...
Pesi caricati con successo!


GPTLightningModule(
  (model): GPTModel(
    (token_embedding_table): Embedding(4002, 256)
    (position_embedding_table): Embedding(128, 256)
    (emb_dropout): Dropout(p=0.1, inplace=False)
    (blocks): Sequential(
      (0): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (ffwd): FeedFoward(
          (net): Sequential(
            (0): Linear(in_features=256, out_features=1024, bias=True)
            (1): ReLU()
            (2): Linear(in_features=1024, out_features=256, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
      (1): Block(
        (mha): MultiheadAttention(
          (out_proj): NonDyn

In [ ]:
import torch
import torch.nn as nn

# 1. Recuperiamo le dimensioni
# Usiamo le proprietà del layer attuale per non sbagliare
vecchio_vocab_size = model.model.lm_head.out_features
nuovo_vocab_size = tokenizer.get_vocab_size()
embed_dim = model.model.lm_head.in_features

print(f"Ridimensionamento: da {vecchio_vocab_size} a {nuovo_vocab_size} token.")

# =====================================================
# 2. AGGIORNO L'EMBEDDING (token_embedding_table)
# =====================================================
nuovo_embedding = nn.Embedding(nuovo_vocab_size, embed_dim)

# Copiamo i pesi esistenti
with torch.no_grad():
    nuovo_embedding.weight[:vecchio_vocab_size] = model.model.token_embedding_table.weight

# Sostituiamo il layer nel sottomodello 'model'
model.model.token_embedding_table = nuovo_embedding

# =====================================================
# 3. AGGIORNO LA TESTA (lm_head)
# =====================================================
# Nota: il tuo modello ha bias=True, quindi lo manteniamo
nuova_lm_head = nn.Linear(embed_dim, nuovo_vocab_size, bias=True)

# Copiamo pesi e bias esistenti
with torch.no_grad():
    nuova_lm_head.weight[:vecchio_vocab_size] = model.model.lm_head.weight
    nuova_lm_head.bias[:vecchio_vocab_size] = model.model.lm_head.bias

# Sostituiamo il layer
model.model.lm_head = nuova_lm_head

print("Operazione completata. Il modello è pronto per il fine-tuning.")

Ridimensionamento: da 4000 a 4002 token.
Operazione completata. Il modello è pronto per il fine-tuning.


In [ ]:
from model.train import run_training

best_model = run_training(
        train_loader = train_dataloader,
        val_loader = val_dataloader,
        save_dir = path_checkpoints,
        experiment_name="dataset_riddles", # Dai nomi significativi!
        model = model,
        max_epochs = 20,
        patience=3 # Ferma se non migliora per 8 epoche
    )

INFO: GPU available: True (cuda), used: True
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable Li

--- Starting Experiment: dataset_riddles ---


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPTModel │  6.8 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 6.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.8 M                                                                                                
Total estimated model params size (MB): 27                                                                         
Modules in train mode: 85                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: Metric val_loss improved. New best score: 4.592
INFO: Metric val_loss improved. New best score: 4.592
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 4.592
INFO: Metric val_loss improved by 0.326 >= min_delta = 0.0. New best score: 4.266
INFO: Metric val_loss improved by 0.326 >= min_delta = 0.0. New best score: 4.266
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.326 >= min_delta = 0.0. New best score: 4.266
INFO: Metric val_loss improved by 0.219 >= min_delta = 0.0. New best score: 4.047
INFO: Metric val_loss improved by 0.219 >= min_delta = 0.0. New best score: 4.047
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.219 >= min_delta = 0.0. New best score: 4.047
INFO: Metric val_loss improved by 0.162 >= min_delta = 0.0. New best score: 3.885
INFO: Metric val_loss improved by 0.162 >= min_delta = 0.0. New best score: 3.885
INFO:lightning.pytorch.callbacks.early_stopping:Metric v

Training completato. Il miglior modello è salvato in:
/content/small_LM/checkpoints/dataset_riddles/gpt-epoch=19-val_loss=3.2507.ckpt


In [ ]:
import torch

# Definisci il percorso sul tuo Drive
# Ti consiglio di creare prima una cartella (es. 'checkpoints') su Drive
path_checkpoint = "/content/drive/MyDrive/PercorsoEccellenza/riddles/40_epoche.ckpt"

# Salva il dizionario dello stato del modello
torch.save(model.state_dict(), path_checkpoint)

print(f"Modello salvato in: {path_checkpoint}")

Modello salvato in: /content/drive/MyDrive/PercorsoEccellenza/riddles/40_epoche.ckpt


## Inference

In [ ]:
from model.lightning_model import GPTLightningModule
print("Caricamento del modello migliore per inferenza...")
best_model = GPTLightningModule.load_from_checkpoint(best_model, vocab_size = tokenizer.get_vocab_size());
best_model.eval(); # Disattiva dropout per inferenza
best_model.to("cuda" if torch.cuda.is_available() else "cpu");

Caricamento del modello migliore per inferenza...


In [ ]:
from tokenizers import Tokenizer

# Carica dal file salvato
tokenizer = Tokenizer.from_file(path + "custom_tokenizer.json")

In [ ]:
id_stop = tokenizer.encode("[EOS]").ids[0]

In [ ]:
import torch

In [ ]:
encoded = tokenizer.encode("[BOS] Tell me a riddle about the time \n RIDDLE:")
input_ids = torch.tensor(encoded.ids)
input_ids = input_ids.reshape(1, -1)

In [ ]:
out = model.generate(input_ids.cuda(), max_new_tokens=300, temperature=0.6, top_k=50)

print(tokenizer.decode(out[0].tolist(), skip_special_tokens= False)+ "\n")

In [ ]:
# 1. Generi un tot di token (es. 100)
out = model.generate(input_ids.cuda(), 300, temperature=0.4, top_k=50, repetition_penalty=1.2)

# 2. Decodifica i token
testo_generato = tokenizer.decode(out[0].tolist(), skip_special_tokens= False)

separatore = "[EOS]" # Usa il token/parola che vuoi forzare
if separatore in testo_generato:
    testo_pulito = testo_generato[5:].split(separatore)[0]
else:
    testo_pulito = testo_generato

print(testo_pulito)

 Tell me a riddle about the time 
 RIDDLE: What goes up and down, up and down, up and down. I am the best of all. I never had any friends or a friend but always. And I'm always here to play with you. 
 ANSWER: day 


Temperature: 0.4, top_k = 50, repetition_penalty= 1.2



 Do you know a riddle about a dog?

 RIDDLE: What is it that wags its tail when you grow up?

 ANSWER: dog



 Tell me a riddle about a dog

 RIDDLE: I'm black and brown, but I have no eyes. My tail is black, has a red beast, and has a long neck. My tail is white and soft and soft. My tail is a very strong and brown. What can you do with it?

 ANSWER: The dog



 Tell me a riddle about the sea

 RIDDLE: I am the best of all. And I never see this sea. And I can be seen, but not in a dark place. The sea is full of water and never lie again.

 ANSWER: beach



Tell me a riddle about Italy

 RIDDLE: I am the only one, I'm the size of my heart. You can fly but not be seen. My heart is in pain, and I can't get out. My heart will never cry, but never move.

 ANSWER: it



Tell me a riddle about some friends

 RIDDLE: What has no eyes, but no one is watching. And bends at the same time?

 ANSWER: sun



 Tell me a riddle about the time

 RIDDLE: What goes up and down, up and down, up and down. I am the best of all. I never had any friends or a friend but always. And I'm always here to play with you.

 ANSWER: day

 Tell me a riddle about music
 RIDDLE: I use to turn it out and make it spin. Ralace up and down, and twelve I see.
 ANSWER: music

In [ ]:
tokenizer.decode((out[0,:6]).tolist())
out[0,7]

tensor(302, device='cuda:0')

Hello, how are you? @ Hi , I just looking for a new winter I think I ’ m looking for a pair of trousers . @ What style do you want to rent ? @ I want to take a pair of trousers for a computer . @ I ’ m looking . @

Hello, how are you? @ I'm looking for a special pair of shoes ? @ I thought I know what you'll recommend it ? @ You'd like to make a pair of shoes for the color . @ Do you like anything else ? How much would you like to buy a